# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/8(목) 오후 · 코드 리뷰 · 테스트 · 디버깅 · 완성과 회고 — 실습</mark>

오전에 명령 한 번으로 도는 파이프라인을 만들었습니다. 오후에는 그 코드를 **점검하고**, 일부러 망가뜨린 코드를 **고치고**, 1과목을 **마무리**합니다.

| 교시 | 무엇 | 만드는 파일 |
|---|---|---|
| 5교시 | 코드 리뷰와 단위 테스트 | `test_agent_core.py` |
| 6교시 | 디버깅 — 에러가 있는 코드 고치기 | — |
| 7교시 | 완성과 회고 · 깃허브에 올리기 | `docs/day08_retrospective.md` |

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저

1. 이 파일을 오전 노트북과 같은 `security-agent-toolkit/agent_core` 폴더에 넣습니다.
2. 오른쪽 위 **커널 선택**에서 오전과 같은 **각자 만든 가상환경**을 고릅니다.
3. 아래 **위치 확인 셀**과 **준비 셀 일곱**을 차례로 실행합니다. 오전을 못 끝냈어도 여기서부터 할 수 있습니다.

### ⚠ 중요 · 무료 키는 1분에 15번까지만 호출할 수 있습니다

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px; border-left:6px solid #e65100">
1. 한도를 넘으면 답을 받지 못합니다. 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
2. 오후에 LLM 을 부르는 문제는 <b>7-2 · 7-3 · ⭐7-6</b> 뿐입니다. 셋 다 한 번 실행에 두 번 부릅니다.<br>
3. 한도에 걸리면 <code>[LLM 호출 실패] 상태 코드 429</code> 가 먼저 보이고, 그 아래에 <code>KeyError: 'choices'</code> 나 <code>TypeError: 'NoneType' object is not subscriptable</code> 같은 에러가 이어질 수 있습니다. <b>코드가 틀린 것이 아닙니다.</b> 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
4. LLM 의 답 문장과 위험도는 실행할 때마다 달라질 수 있습니다. 🎯 와 글자가 같지 않아도 <b>모양이 같으면</b> 맞습니다.
</div>

### 0.2 문제를 푸는 법

1. 문제 앞의 **🐍 문법 상자**를 먼저 읽습니다.
2. 6교시 문제에는 **에러가 있는 코드가 들어 있습니다.** 먼저 그대로 실행해 에러를 보고, 고쳐서 다시 실행합니다.
3. 막히면 **💡 힌트**를 보고, 그래도 막히면 노트북 맨 아래 「정답」을 봅니다. 도전 문제는 안 풀고 넘어가도 됩니다.
4. `NameError` 가 나오면 위쪽 셀을 건너뛴 것입니다.

In [ ]:
# 위치 확인 셀 — 그대로 실행합니다
import os                                                     # 폴더 · 파일을 다루는 도구
import sys                                                    # 지금 실행 중인 파이썬의 정보를 주는 도구


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                                        # 다섯 칸 위까지 찾는다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 리턴한다
        folder = os.path.dirname(folder)                      # 한 칸 위 폴더로 올라간다
    return None                                               # None 을 리턴하고 여기서 끝낸다


ENV_PATH = find_env()                                         # 찾은 .env 경로. 못 찾으면 None

print("노트북이 실행되는 폴더:", os.getcwd())                           # 노트북이 도는 폴더
print("지금 쓰는 파이썬     :", sys.executable)                      # 커널로 고른 파이썬 — 내 가상환경이어야 한다

if ENV_PATH:                                                  # .env 를 찾았으면
    print("키 파일(.env)         :", ENV_PATH)                   # 찾은 .env 의 위치
    print("준비 완료")
else:                                                         # 아니면
    print(".env 를 찾지 못했습니다")
    print("1) 이 노트북을 새 security-agent-toolkit 폴더 안의 agent_core 폴더로 옮긴 뒤 다시 엽니다")
    print("2) 아침 과제 6 의 「.env 옮기기」를 했는지 봅니다")

아래 일곱 셀도 실행합니다. 어제와 오전에 만든 파일을 정답과 같은 내용으로 다시 만들고(이미 있으면 같은 내용으로 덮어씁니다), 오늘 쓰는 데이터와 설정을 저장합니다.

In [ ]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 리턴한다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None                                               # None 을 리턴하고 여기서 끝낸다


def read_api_key():                                           # .env 에서 키를 읽어 리턴하는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 리턴한다


def call_llm(question):                                       # 질문을 받아 답 문장을 리턴하는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 요청 본문 — 모델 · temperature 0 · 질문
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}
    # POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 리턴한다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 리턴한다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다

    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 리턴한다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None                                           # None 을 리턴하고 여기서 끝낸다

In [ ]:
%%writefile event_summarizer.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

SIZE = 6                                                      # 한 묶음에 넣을 경보 수
# 요약 지시 — 출력 형식(JSON 배열)과 키 이름을 정한다
instruction = "다음 보안 경보 목록을 읽고, 경보마다 id, risk_level, summary 세 키를 가진 JSON 배열로만 답하세요. "
# 위험도 세 단어와 요약 길이를 정한다. 뒤에 경보 목록이 붙는다
instruction = instruction + "risk_level 은 high, medium, low 중 하나입니다. summary 는 한국어 한 문장으로 짧게 씁니다. 경보 목록: "


def summarize_batch(batch):                                   # 묶음 하나를 LLM 으로 요약하는 함수
    # 묶음을 JSON 문자열로 바꿔 지시와 함께 보낸다
    text = call_llm(instruction + json.dumps(batch, ensure_ascii=False))
    result = parse_llm_json(text)                             # 답을 딕셔너리로 바꾼다. 못 읽으면 None
    if result is None:                                        # JSON 으로 읽지 못했으면
        return []                                             # 실패하면 빈 리스트 — 다음 묶음은 계속한다
    return result                                             # 요약 리스트를 리턴한다


def summarize_events(events):                                 # 경보 전체를 묶음으로 나눠 요약하는 함수
    summaries = []                                            # 요약을 모을 빈 리스트
    for start in range(0, len(events), SIZE):                 # 0, 6, 12 … 묶음의 시작 번호
        # 묶음 하나를 요약해 한 건씩 꺼낸다
        for item in summarize_batch(events[start:start + SIZE]):
            summaries.append(item)                            # 요약 하나를 모은다
    return summaries                                          # 모은 요약을 리턴한다


def sort_by_risk(summaries):                                  # 요약을 high → medium → low 순서로 정렬하는 함수
    sorted_summaries = []                                     # 위험도순으로 모을 빈 리스트
    for level in ["high", "medium", "low"]:                   # 이 순서대로 모은다
        for s in summaries:                                   # 요약을 하나씩 꺼낸다
            if s["risk_level"].lower() == level:              # 소문자로 맞춰 지금 위험도와 같으면
                sorted_summaries.append(s)                    # 정렬 결과에 넣는다
    return sorted_summaries                                   # 정렬한 요약을 리턴한다

In [ ]:
%%writefile report_generator.py
from llm_client import call_llm                               # llm_client.py 의 call_llm 을 꺼내 쓴다

# 총평 지시 — 숫자는 코드가 세므로 쓰지 말라고 한다
overview_instruction = "다음은 밤사이 보안 경보 요약입니다. 팀장에게 보고할 총평을 한국어 두세 문장으로 쓰세요. 숫자는 쓰지 마세요. 요약: "


def make_lines(sorted_summaries):                             # 정렬된 요약을 목록 줄로 만드는 함수
    lines = ""                                                # 목록 줄을 모을 빈 문자열
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        # 「- [HIGH] E01 요약」 한 줄을 붙인다
        lines = lines + f"- [{s['risk_level'].upper()}] {s['id']} {s['summary']}\n"
    return lines                                              # 목록 줄을 리턴한다


def make_overview(sorted_summaries):                          # 정렬된 요약으로 총평을 받는 함수
    # 요약 줄을 지시와 함께 보내 총평을 받는다
    text = call_llm(overview_instruction + make_lines(sorted_summaries))
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        return "총평을 만들지 못했습니다. 건별 내역을 먼저 확인하세요."              # 실패를 숨기지 않고 알린다
    return text.strip()                                       # 앞뒤 공백을 지운 총평을 리턴한다


def build_report(sorted_summaries, overview, today):          # 틀 · 숫자는 코드로, 총평은 받은 문장으로 보고서를 만드는 함수
    total = len(sorted_summaries)                             # 전체 건수 — 코드가 센다
    high_count = 0                                            # high 건수. 0 에서 시작
    for s in sorted_summaries:                                # 정렬된 요약을 하나씩 꺼낸다
        if s["risk_level"].lower() == "high":                 # 소문자로 맞춰 high 인지 본다
            high_count = high_count + 1                       # high 하나를 센다

    warning = ""                                              # 기본은 경고 없음
    if high_count >= 3:                                       # high 가 3건 이상이면
        # 경고 줄 — 보고서 맨 위 「한눈에 보기」에 들어간다
        warning = f"- **주의: high 경보 {high_count}건 — 건별 내역을 먼저 확인할 것**\n"

    report = f"# 야간 보안 관제 보고 ({today})\n\n"                   # 제목 — 날짜만 바뀐다
    # 건수 줄과 경고 줄 — 코드가 센 값
    report = report + "## 한눈에 보기\n" + f"- 처리한 경보: {total}건 (high {high_count}건)\n" + warning
    report = report + "\n## 총평\n" + overview + "\n"           # 총평 — LLM 이 쓴 문장
    # 정렬한 목록 줄
    report = report + "\n## 건별 내역 (위험한 것부터)\n" + make_lines(sorted_summaries)
    return report                                             # 완성한 보고서 문자열을 리턴한다


def save_report(report, today):                               # 보고서를 날짜가 든 파일로 저장하는 함수
    filename = f"daily_report_{today.replace('-', '')}.md"    # 날짜의 - 를 빼서 파일 이름을 만든다 — daily_report_20261007.md
    with open(filename, "w", encoding="utf-8") as f:          # 보고서 파일을 쓰기로 연다 (있으면 덮어쓴다)
        f.write(report)                                       # 보고서 문자열을 파일에 쓴다
    return filename                                           # 저장한 파일 이름을 리턴한다

In [ ]:
%%writefile notifier.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# 꼭 있어야 하는 키 네 개
REQUIRED = ["model", "approve_severity", "report_folder", "webhook_url"]
RANK = {"low": 1, "medium": 2, "high": 3}                     # 위험도 순위 — 클수록 위험하다


def load_config(path):                                        # 설정을 읽고 빠진 키가 없는지 검사하는 함수
    with open(path, encoding="utf-8") as f:                   # 설정 파일을 읽기로 연다
        config = json.load(f)                                 # 설정을 딕셔너리로 읽는다
    for key in REQUIRED:                                      # 꼭 있어야 하는 키를 하나씩
        if key not in config:                                 # 설정에 그 키가 없으면
            print("[설정 오류] 빠진 키:", key)                       # 어떤 키가 빠졌는지 알린다
            return None                                       # None 을 리턴하고 여기서 끝낸다
    return config                                             # 다 있으면 설정을 리턴한다


def needs_approval(severity, config):                         # 사람에게 확인받을 위험도인지 판정하는 함수
    level = RANK.get(severity.lower(), 3)                     # 모르는 위험도는 high 로 본다 — 놓치는 것보다 묻는 게 낫다
    return level >= RANK[config["approve_severity"]]          # 기준 이상이면 True


def notify(message, url):                                     # 메시지를 알림 서버로 보내는 함수
    try:                                                      # 아래 줄을 해 본다
        requests.post(url, json={"text": message}, timeout=5)  # 메시지를 POST 한다 — 5초만 기다린다
        return True                                           # 보냈다
    except Exception as e:                                    # 에러가 나면 e 에 담아 여기로 온다
        print("[알림 실패]", type(e).__name__)                    # 에러 이름을 알린다 — 숨기지 않는다
        return False                                          # 못 보냈다 — 부른 쪽은 계속 간다

In [ ]:
%%writefile pipeline.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
from datetime import date                                     # 오늘 날짜를 얻는 도구

import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import event_summarizer                                       # 오전에 만든 event_summarizer.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다


def run_report(events, today):                                # 요약 → 정렬 → 총평 → 저장을 묶은 함수
    summaries = event_summarizer.summarize_events(events)     # 묶음으로 나눠 요약한다 — LLM
    # 위험도순으로 정렬한다
    sorted_summaries = event_summarizer.sort_by_risk(summaries)
    # 총평을 받는다 — LLM 한 번
    overview = report_generator.make_overview(sorted_summaries)
    # 틀 · 숫자 · 총평으로 보고서를 만든다
    report = report_generator.build_report(sorted_summaries, overview, today)
    filename = report_generator.save_report(report, today)    # 날짜가 든 파일로 저장한다
    # 결과 둘을 딕셔너리 하나로 리턴한다
    return {"filename": filename, "summaries": sorted_summaries}


def run_pipeline(config_path, today):                         # 설정 → 보고서 → 세기 → 알림을 차례로 부르는 함수
    config = notifier.load_config(config_path)                # 설정을 검사하며 읽는다
    if config is None:                                        # 설정에 빠진 키가 있으면
        print("[중단] config.json 을 고친 뒤 다시 실행하세요")             # 왜 멈췄는지 알린다
        return                                                # 더 가지 않고 함수를 끝낸다

    llm_client.MODEL = config["model"]                        # 모델 이름도 설정 파일에서 온다
    with open("events_1008.json", encoding="utf-8") as f:     # 오늘 처리할 경보 파일을 연다
        events = json.load(f)                                 # 경보 목록을 리스트로 읽는다
    result = run_report(events, today)                        # 보고서를 만든다 — LLM 두 번

    approve = 0                                               # 사람 확인이 필요한 건수. 0 에서 시작
    for s in result["summaries"]:                             # 정렬된 요약을 하나씩
        if notifier.needs_approval(s["risk_level"], config):  # 사람에게 확인받을 위험도면
            approve = approve + 1                             # 하나 센다

    # 알림으로 보낼 한 줄
    message = f"[보고서] {result['filename']} 저장 · 사람 확인 필요 {approve}건"
    notifier.notify(message, config["webhook_url"])           # 알림 — 실패해도 멈추지 않는다
    print(message)                                            # 보낸 알림을 화면에도 보여 준다


if __name__ == "__main__":                                    # python pipeline.py 로 직접 실행할 때만
    run_pipeline("config.json", str(date.today()))            # 오늘 날짜로 처음부터 끝까지 돌린다

In [ ]:
%%writefile alert_server.py
from flask import Flask, request                              # 10/2 에 쓴 웹 서버 도구

app = Flask(__name__)                                         # 서버를 하나 만든다


@app.route("/alert", methods=["POST"])                        # /alert 로 POST 가 오면 아래 함수가 받는다
def alert():                                                  # 알림을 받는 함수
    data = request.get_json()                                 # 보낸 JSON 을 딕셔너리로 꺼낸다
    print("[알림 받음]", data["text"])                         # 터미널에 알림 내용을 보여 준다
    return {"ok": True}                                       # 받았다고 답한다 (상태 코드 200)


app.run(port=5001)                                            # 5001 번 포트에서 기다린다

In [ ]:
# 준비 셀 — 그대로 실행합니다. 오늘 쓰는 경보 6건과 설정 파일 둘을 저장합니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

events_1008 = [                                               # 7교시 「다른 날」로 쓸 경보 6건
    {"id": "F01", "rule": "brute_force", "target": "admin", "time": "02:10", "detail": "203.0.113.9 에서 admin 로그인 실패 7회 뒤 성공"},
    {"id": "F02", "rule": "failed_login", "target": "kim01", "time": "09:05", "detail": "kim01 로그인 실패 1회"},
    {"id": "F03", "rule": "failed_login", "target": "lee.yh", "time": "09:40", "detail": "lee.yh 로그인 실패 1회 뒤 바로 성공"},
    {"id": "F04", "rule": "config_changed", "target": "firewall", "time": "18:00", "detail": "점검 시간에 ops.kim 이 방화벽 규칙 1개 수정"},
    {"id": "F05", "rule": "failed_login", "target": "park.js", "time": "10:12", "detail": "park.js 로그인 실패 1회"},
    {"id": "F06", "rule": "new_ip_login", "target": "jung.hw", "time": "08:55", "detail": "jung.hw 가 사내 다른 층 IP 에서 로그인"},
]

config = {                                                    # 오전 2교시의 설정 — 키 네 개
    "model": "gemini-3.5-flash-lite",
    "approve_severity": "high",
    "report_folder": "reports",
    "webhook_url": "http://127.0.0.1:5001/alert",
}
# 키 두 개가 빠진 설정
broken = {"model": "gemini-3.5-flash-lite", "approve_severity": "high"}

with open("events_1008.json", "w", encoding="utf-8") as f:        # 오늘 처리할 경보
    json.dump(events_1008, f, ensure_ascii=False, indent=2)   # 내일 경보를 파일에 저장한다
with open("config.json", "w", encoding="utf-8") as f:             # 오전 2교시의 설정 — 키 네 개
    json.dump(config, f, ensure_ascii=False, indent=2)        # 설정을 사람이 읽기 좋게 저장한다
with open("config_broken.json", "w", encoding="utf-8") as f:      # 키 두 개가 빠진 설정 — 7교시에 쓴다
    json.dump(broken, f, ensure_ascii=False, indent=2)        # 빠진 설정을 저장한다

print(f"경보 {len(events_1008)}건 · 설정 파일 2개를 저장했습니다")           # 저장한 것을 알린다

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · 코드 리뷰와 단위 테스트</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **코드 리뷰 (code review)** | 남의 코드를 볼 때 무엇부터 확인하나 |
| **단위 테스트 (unit test)** | 함수 하나가 맞게 도는지 자동으로 확인하는 방법 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 코드 리뷰 →
- 단위 테스트 →

### 왜 필요한가

1. 코드가 **돌아간다고 맞는 것은 아닙니다.** 키가 코드에 적혀 있거나, 실패를 숨기는 코드도 잘 돕니다.
2. **코드 리뷰** — 사람이 체크리스트로 한 줄씩 읽습니다. 매번 같은 항목을 보니 빠뜨리지 않습니다.
3. **단위 테스트** — `assert` 로 「이 입력이면 이 값이 나와야 한다」를 적어 둡니다. 코드를 고친 뒤 한 번 돌리면 망가진 곳을 바로 압니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 리뷰 체크리스트 — 이 코드는 다섯 항목 가운데 어디를 어기나</mark>

```python
from llm_client import read_api_key, parse_llm_json

api_key = read_api_key()                                                # (가)
body = {"model": "gemini-3.5-flash-lite", "messages": messages}         # (나)
response = requests.post(URL, headers=headers, json=body)               # (다)
judgment = parse_llm_json(response.json()["choices"][0]["message"]["content"])   # (라)

try:
    requests.post("http://127.0.0.1:5001/alert", json={"text": "완료"}, timeout=5)
except Exception:
    pass                                                                # (마)
```

| 항목 | 확인할 것 | 볼 줄 |
|---|---|---|
| 비밀 | API 키가 코드에 직접 적혀 있지 않은가 | (가) |
| 설정 | 모델 이름 · 주소가 `config.json` 에서 오는가 | (나) |
| 검증 | LLM 답을 그대로 믿지 않고 확인하는가 | (라) |
| 실패 | 요청에 `timeout` 이 있는가 | (다) |
| 기록 | 실패를 숨기지 않고 알리는가 | (마) |

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
assert len("high") == 4, "길이가 다르다"
print("첫 번째 검사 통과")

assert "high".upper() == "HIGH", "대문자가 아니다"
print("두 번째 검사 통과")
```

막히면 바로 위 `5.1 리뷰 체크리스트 — 이 코드는 다섯 항목 가운데 어디를 어기나` 설명을 다시 봅니다.

In [ ]:
assert len("high") == 4, "길이가 다르다"
print("첫 번째 검사 통과")

assert "high".upper() == "HIGH", "대문자가 아니다"
print("두 번째 검사 통과")

✅ `첫 번째 검사 통과` · `두 번째 검사 통과` — 두 조건이 다 참이라 `assert` 는 아무것도 보여 주지 않고 지나갑니다

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 체크리스트로 코드 리뷰하기</font></h3></td></tr></table>

위 5.1 의 코드를 다섯 항목으로 리뷰해, 항목마다 「통과」 또는 「위반」 을 `review` 에 담고 위반 수를 세시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `비밀 통과` · `설정 위반` · `검증 통과` · `실패 위반` · `기록 위반` 과 `위반 3 개` |

**💡 힌트**

1. (나)의 모델 이름은 어디서 오나요?
2. (다)에 `timeout` 이 있나요?
3. (마)의 `pass` 는 실패했을 때 무엇을 보여 주나요?

In [2]:
# ― 미리 채워 둔 줄입니다. 고치지 않습니다 ―
checklist = ["비밀", "설정", "검증", "실패", "기록"]       # 위 표의 다섯 항목
# ― 여기까지 ―

# 1. review 라는 딕셔너리를 만들어, 다섯 항목마다 "통과" 또는 "위반" 을 담으세요
review = {
    "비밀": "통과",
    "설정": "위반",
    "검증": "통과",
    "실패": "위반",
    "기록": "위반"
}

for item in checklist:                                 # 체크리스트 항목을 하나씩
    print(item, review[item])                          # 항목과 판정

# 2. 「위반」 이 몇 개인지 세어 「위반 3개」 처럼 출력하세요
violation_count = sum(1 for status in review.values() if status == "위반")
print(f"위반 {violation_count}개")

비밀 통과
설정 위반
검증 통과
실패 위반
기록 위반
위반 3개


#### 🐍 문법 상자 · `assert` — 참이면 조용히, 거짓이면 멈춘다

```python
assert 1 + 1 == 2, "1 + 1 은 2 여야 한다"     # 참 — 아무것도 보이지 않고 다음 줄로
print("첫 번째 검사 통과")
assert 1 + 1 == 3, "1 + 1 은 2 여야 한다"     # 거짓 — 여기서 멈춘다
print("두 번째 검사 통과")                  # 실행되지 않는다

# 첫 번째 검사 통과
# AssertionError: 1 + 1 은 2 여야 한다
```

| 쓰는 것 | 결과 |
|---|---|
| `assert 조건, "메시지"` | 조건이 참이면 통과, 거짓이면 `AssertionError: 메시지` 로 멈춘다 |
| `assert 함수(입력) == 기대값` | 함수가 기대한 값을 리턴하는지 확인한다 — 이것이 테스트다 |

⚠ 메시지를 빼면 `AssertionError` 만 보여서 무엇이 틀렸는지 모릅니다. 메시지에 「무엇이 어때야 하는지」를 씁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · 첫 테스트 — make_lines</font></h3></td></tr></table>

어제 만든 `report_generator.make_lines` 가 맞게 도는지 `assert` 세 줄로 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[테스트 통과] make_lines 3건` |

**💡 힌트**

1. 모양은 `assert 함수(입력) == 기대값, "메시지"` 입니다.
2. `make_lines` 는 위험도를 대문자로 바꾸고 줄 끝에 `\n` 을 붙입니다.
3. `AssertionError` 가 나면 메시지를 읽고 기대값의 글자를 하나씩 비교합니다.

In [3]:
import importlib                                              # 파일을 다시 읽는 도구
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다

importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
sample = [{"id": "E01", "risk_level": "High", "summary": "로그인 실패 4회"}]   # 대문자가 섞인 요약 한 건
two = [{"id": "E02", "risk_level": "low", "summary": "새 IP 로그인"}, {"id": "E03", "risk_level": "medium", "summary": "심야 접속"}]   # 요약 두 건
# ── 여기까지 ──

# 1. report_generator.make_lines(sample) 이 "- [HIGH] E01 로그인 실패 4회\n" 과 같은지 assert 로 확인하세요. 메시지는 "make_lines 결과가 다르다"
assert report_generator.make_lines(sample) == "- [HIGH] E01 로그인 실패 4회\n", "make_lines 결과가 다르다"


# 2. 빈 리스트 [] 를 넣으면 빈 문자열 "" 이 나오는지 assert 로 확인하세요. 메시지는 "빈 리스트는 빈 문자열이어야 한다"
assert report_generator.make_lines([]) == "", "빈 리스트는 빈 문자열이어야 한다"

# 3. 요약 두 건 two 를 넣으면 줄이 두 개(\n 이 두 개)인지 assert 로 확인하세요. 메시지는 "한 건에 한 줄이어야 한다"
assert report_generator.make_lines(two).count("\n") == 2, "한 건에 한 줄이어야 한다"

print("[테스트 통과] make_lines 3건")                               # 여기까지 오면 assert 가 모두 참이었다

[테스트 통과] make_lines 3건


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · needs_approval 테스트</font></h3></td></tr></table>

오전에 만든 `notifier.needs_approval` 를 세 경우로 테스트하시오. 처음 보는 위험도도 사람에게 묻는지 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[테스트 통과] needs_approval 3건` |

**💡 힌트**

1. `critical` 은 `RANK` 에 없는 위험도입니다.
2. `notifier.py` 는 모르는 위험도를 high 로 봅니다(`RANK.get(…, 3)`).
3. 놓치는 것보다 묻는 게 낫습니다. 보안에서는 모를 때 막는 쪽을 고릅니다.

In [4]:
import importlib                                              # 파일을 다시 읽는 도구
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
config = {"approve_severity": "high"}                       # 기준만 있으면 판정할 수 있다
# ── 여기까지 ──

# 1. needs_approval("high", config) 가 True 인지 assert 로 확인하세요
assert notifier.needs_approval("high", config) == True, "high 위험도는 승인 필요"

# 2. needs_approval("low", config) 가 False 인지 assert 로 확인하세요
assert notifier.needs_approval("low", config) == False, "low 위험도는 승인 불필요"

# 3. 처음 보는 위험도 "critical" 이 True 인지 assert 로 확인하세요
assert notifier.needs_approval("critical", config) == True, "critical 위험도는 승인 필요"

print("[테스트 통과] needs_approval 3건")                           # 여기까지 오면 assert 가 모두 참이었다

[테스트 통과] needs_approval 3건


#### 🐍 문법 상자 · `None` 인지는 `is None` 으로 봅니다

```python
result = None
empty = ""
print(result is None, empty is None, result == empty)
# True False False
```

| 값 | `is None` | 뜻 |
|---|---|---|
| `None` | `True` | 값이 없다 — `parse_llm_json` 이 실패하면 리턴한다 |
| `""` | `False` | 빈 문자열 — `call_llm` 이 실패하면 리턴한다 |

⚠ `None` 과 `""` 는 다른 값입니다. 어느 함수가 무엇을 리턴하는지 보고 비교합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · parse_llm_json 테스트</font></h3></td></tr></table>

10/6 에 만든 `llm_client.parse_llm_json` 이 코드 블록에 싸인 JSON 은 딕셔너리로, 깨진 문장과 빈 문자열은 `None` 으로 리턴하는지 테스트하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[테스트 통과] parse_llm_json 3건` |

**💡 힌트**

1. 딕셔너리끼리도 `==` 로 비교합니다.
2. `None` 인지는 `is None` 으로 봅니다.
3. LLM 은 가끔 JSON 대신 문장을 보냅니다. 그때 파이프라인이 죽지 않는지가 이 테스트입니다.

In [5]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                                  # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
fenced = "```json\n{\"tool\": \"lock_account\"}\n```"          # LLM 이 자주 보내는 모양 — 코드 블록에 싸인 JSON
# ── 여기까지 ──

# 1. parse_llm_json(fenced) 가 {"tool": "lock_account"} 와 같은지 assert 로 확인하세요
assert llm_client.parse_llm_json(fenced) == {"tool": "lock_account"}, "parse_llm_json 결과가 다르다"

# 2. parse_llm_json("그럴듯한 문장입니다") 가 None 인지 assert 로 확인하세요 (is None)
assert llm_client.parse_llm_json("그럴듯한 문장입니다") is None, "그럴듯한 문장은 None 이어야 한다"

# 3. parse_llm_json("") 가 None 인지 assert 로 확인하세요
assert llm_client.parse_llm_json("") is None, "빈 문자열은 None 이어야 한다"

print("[테스트 통과] parse_llm_json 3건")                           # 여기까지 오면 assert 가 모두 참이었다

[테스트 통과] parse_llm_json 3건


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-6 · 테스트를 파일 하나로 모아 터미널에서 돌리기</font></h3></td></tr></table>

5-3 ~ 5-5 의 `assert` 아홉 줄을 **`test_agent_core.py`** 로 모으시오. 셀을 실행한 뒤 💻 터미널(`agent_core` 폴더)에 `python test_agent_core.py` 를 입력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 셀은 `Writing test_agent_core.py`, 터미널에 `[테스트 통과] 9건 모두` |

**💡 힌트**

1. 각 문제의 미리 채운 데이터(`sample` · `two` · `config` · `fenced`)도 함께 옮깁니다.
2. 파일 맨 위에서 세 파일을 `import` 합니다.
3. 앞으로 코드를 고칠 때마다 이 명령 한 번으로 망가진 곳이 없는지 봅니다.

In [6]:
%%writefile test_agent_core.py
import llm_client                                      # 오전에 만든 llm_client.py 를 불러온다
import notifier                                        # 3교시에 만든 notifier.py 를 불러온다
import report_generator                                # 오후에 만든 report_generator.py 를 불러온다

# 1. 문제 5-3 의 assert 세 줄을 옮기세요 (sample · two 도 함께)
sample = {"title": "경보 테스트", "severity": "high", "risk_level": "high"}
two = [sample, sample]
assert notifier.needs_approval("high", {"approve_severity": "medium"}) == True
assert notifier.needs_approval("low", {"approve_severity": "medium"}) == False
assert len(two) == 2

# 2. 문제 5-4 의 assert 세 줄을 옮기세요 (config 도 함께)
config = {"model": "gpt-4o", "approve_severity": "high", "report_folder": "reports", "webhook_url": "http://127.0.0.1:5999/alert"}
assert notifier.load_config("config.json") is not None
assert config["model"] == "gpt-4o"
assert config["approve_severity"] in ["low", "medium", "high"]

# 3. 문제 5-5 의 assert 세 줄을 옮기세요 (fenced 도 함께)
fenced = "```json\n{\"test\": True}\n```"
assert report_generator.build_report([], "총평 테스트", "2026-10-08") is not None
assert "2026-10-08" in report_generator.build_report([], "총평 테스트", "2026-10-08")
assert llm_client.strip_fenced_code(fenced) == "{\"test\": True}"

print("[테스트 통과] 9건 모두")                        # 여기까지 오면 아홉 줄이 모두 참이었다

Writing test_agent_core.py


### 📋 5교시 한눈에

| 방법 | 하는 일 | 쓰는 것 |
|---|---|---|
| 코드 리뷰 | 매번 같은 항목으로 코드를 읽는다 | 비밀 · 설정 · 검증 · 실패 · 기록 |
| 단위 테스트 | 입력과 기대값을 적어 두고 자동으로 확인한다 | `assert 함수(입력) == 기대값, "메시지"` |
| 테스트 파일 | 고칠 때마다 한 번에 확인한다 | `python test_agent_core.py` |

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · 디버깅 — 에러가 있는 코드 고치기</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **디버깅 (debugging)** | 에러의 원인을 찾는 순서 |
| **트레이스백 (traceback)** | 파이썬 에러 메시지에서 무엇을 먼저 읽나 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 디버깅 →
- 트레이스백 →

### 왜 필요한가

1. 지금까지 에러가 나면 정답을 봤습니다. 캡스톤에서는 정답이 없습니다. 원인은 **에러 메시지**에 적혀 있습니다.
2. 에러는 세 종류입니다 — 문법 에러(실행 전에 막힌다), 실행 에러(실행 중에 멈춘다), **논리 에러(멈추지 않고 틀린 결과를 낸다)**.
3. 6교시 문제는 모두 **에러가 있는 코드**로 시작합니다. 먼저 그대로 실행해 무엇이 보이는지 읽고, 고칩니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 에러가 나면 마지막 줄부터 읽습니다</mark>

| 순서 | 할 일 |
|---|---|
| 1 | **마지막 줄**의 에러 이름과 내용을 읽는다 — `KeyError: 'sevrity'` |
| 2 | 그 위의 **줄 번호**로 가서 그 줄을 본다 |
| 3 | 에러가 없는데 결과가 이상하면 의심 가는 줄 위에 `print("[디버그]", 값)` 을 넣는다 |
| 4 | 고친 뒤 **다시 실행**해 기대한 결과와 비교한다 |

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
levels = ["High", "high", "low"]

count = 0
for level in levels:
    if level == "high":
        count = count + 1

print(count)
```

막히면 바로 위 `6.1 에러가 나면 마지막 줄부터 읽습니다` 설명을 다시 봅니다.

In [ ]:
levels = ["High", "high", "low"]

count = 0
for level in levels:
    if level == "high":
        count = count + 1

print(count)

✅ `1` — 에러는 없지만 `"High"` 를 세지 못했습니다. 이것이 논리 에러입니다

#### 🐍 문법 상자 · 에러의 세 종류

```python
def make_title(today)          # 문법 에러 (syntax error) — 한 줄도 실행되지 않는다
print(judgment["sevrity"])     # 실행 에러 (runtime error) — 위 줄까지는 돌고 여기서 멈춘다
if level == "high":            # 논리 에러 (logic error) — 에러 없이 틀린 결과가 나온다
```

| 종류 | 에러 메시지 | 찾는 법 |
|---|---|---|
| 문법 에러 (syntax error) | `SyntaxError` — 실행 전에 나온다 | `^` 표시가 가리키는 자리를 본다 |
| 실행 에러 (runtime error) | `KeyError` · `NameError` 등 | 마지막 줄의 에러 이름과 줄 번호를 본다 |
| 논리 에러 (logic error) | 없음 | `print` 로 실제 값을 찍어 기대와 비교한다 |

⚠ 세 번째가 가장 위험합니다. 에러가 없어서 틀린 보고서가 그대로 나갑니다. 그래서 5교시에 테스트를 붙였습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 문법 에러 — 고치면 또 나오는 에러</font></h3></td></tr></table>

아래 셀은 한 줄도 실행되지 않습니다. 에러 메시지를 읽고 고치시오. **하나를 고치면 다음 에러가 나옵니다.** 다 고친 뒤, 두 함수가 맞게 도는지 `assert` 두 줄로 확인합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `# 야간 보안 관제 보고 (2026-10-08)` 그리고 `[테스트 통과] 제목 · 파일 이름 2건` |

**💡 힌트**

1. 파이썬은 문법 에러를 **한 번에 하나씩** 알려 줍니다. 고치고 다시 실행해 다음 에러를 봅니다.
2. `SyntaxError: expected ':'` 는 줄 끝에 빠진 기호, `'(' was never closed` 는 닫지 않은 괄호입니다.
3. `assert` 모양은 5교시와 같습니다 — `assert 함수(입력) == 기대값, "메시지"`.


In [ ]:
# 1. 이 셀을 그대로 실행하고, 에러 메시지의 마지막 줄과 줄 번호를 읽어 고치세요. 다시 실행해 다음 에러도 고치세요

def make_title(today):                                 # 날짜가 든 보고서 제목을 만드는 함수
    return f"# 야간 보안 관제 보고 ({today})"


def make_filename(today):                              # 날짜로 보고서 파일 이름을 만드는 함수
    return "daily_report_" + today.replace("-", "") + ".md"  # 2026-10-08 -> daily_report_20261008.md


print(make_title("2026-10-08"))                        # 제목을 만들어 본다

# 2. make_title 과 make_filename 이 맞는 값을 리턴하는지 assert 두 줄로 확인하세요
assert make_title("2026-10-08") == "# 야간 보안 관제 보고 (2026-10-08)"
assert make_filename("2026-10-08") == "daily_report_20261008.md"

print("[테스트 통과] 제목 · 파일 이름 2건")                 # assert 가 모두 통과하면 이 줄이 나온다

SyntaxError: expected ':' (3734376261.py, line 3)

#### 🐍 문법 상자 · 에러 메시지는 아래에서 위로 읽습니다

함수 안에서 에러가 나면 트레이스백(Traceback · 트레이스백 · 에러가 난 길)이 **두 덩어리**로 나옵니다. 노트북에서 실제로 나오는 모양입니다.

```
KeyError                                  Traceback (most recent call last)
Cell In[1], line 8                        ← ① 내가 함수를 부른 줄
      7 for j in judgments:
----> 8     lines = lines + make_line(j) + "\n"

Cell In[1], line 2, in make_line(j)       ← ② 함수 안에서 실제로 멈춘 줄
      1 def make_line(j):
----> 2     return f"- [{j['sevrity']}] {j['rule']} → {j['tool']}"

KeyError: 'sevrity'                       ← ③ 무엇이 틀렸나 — 여기부터 읽는다
```

| 읽는 순서 | 보는 것 | 위 예에서 |
|---|---|---|
| ③ 맨 아래 줄 | 에러 이름과 문제가 된 값 | `KeyError: 'sevrity'` — 그런 키가 없다 |
| ② 바로 위 덩어리 | **실제로 멈춘 줄** — 대개 여기를 고친다 | `make_line` 의 2번째 줄 |
| ① 맨 위 덩어리 | 그 함수를 부른 줄 | 반복문의 8번째 줄 — 여기는 잘못이 없다 |

| 마지막 줄 | 뜻 | 먼저 볼 곳 |
|---|---|---|
| `KeyError: 'sevrity'` | 딕셔너리에 그 키가 없다 | 키 이름의 철자 |
| `NameError: name 'x' is not defined` | 그 이름을 만든 적이 없다 | 그 줄이 있는 **함수 안에서** 만들었거나 받았는지 |
| `SyntaxError: expected ':'` | 문법이 틀렸다 | `^` 가 가리키는 자리 |

⚠ `----> 8` 의 화살표가 그 덩어리에서 멈춘 줄입니다. `Cell In[1], line 8` 은 셀 안의 8번째 줄이라는 뜻입니다.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · 실행 에러 ① — 함수 안에서 멈추는 코드</font></h3></td></tr></table>

판단 두 건을 보고서 줄로 바꾸는 코드가 멈춥니다. 트레이스백의 **두 덩어리**를 읽어 실제로 틀린 줄을 찾아 고치고, `assert` 두 줄로 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `- [high] brute_force → lock_account` · `- [medium] night_login → notify_only` 두 줄, 그리고 `[테스트 통과] make_line 2건` |

**💡 힌트**

1. 트레이스백 맨 아래 줄부터 읽고, 바로 위 덩어리(`in make_line`)의 `---->` 줄로 갑니다(바로 위 문법 상자).
2. 반복문 줄은 잘못이 없습니다. 함수 안의 키 이름을 딕셔너리와 비교합니다.
3. 첫 판단의 줄이 맞는지, `lines` 가 두 줄인지 `assert` 로 확인합니다. 줄 수는 `lines.count("\n")` 으로 셉니다.


In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
judgments = [                                                 # LLM 판단 두 건이라고 가정합니다
    {"rule": "brute_force", "severity": "high", "tool": "lock_account"},
    {"rule": "night_login", "severity": "medium", "tool": "notify_only"},
]
# ── 여기까지 ──


def make_line(j):                                             # 판단 한 건을 보고서 한 줄로 바꾸는 함수
    return f"- [{j['sevrity']}] {j['rule']} → {j['tool']}"     # 위험도 · 룰 · 도구를 한 줄로


lines = ""                                                    # 보고서 줄을 모을 빈 문자열
for j in judgments:                                           # 판단을 하나씩
    lines = lines + make_line(j) + "\n"                       # 한 줄씩 이어 붙인다
print(lines)                                                  # 모은 줄을 출력한다

# 1. 이 셀을 그대로 실행하고, 트레이스백에서 실제로 멈춘 줄을 찾아 고치세요
# 2. 첫 판단을 make_line 에 넣은 결과가 "- [high] brute_force → lock_account" 인지 assert 로 확인하세요
# 3. lines 에 줄이 두 개(\n 이 두 개) 있는지 assert 로 확인하세요


print("[테스트 통과] make_line 2건")                                # assert 가 모두 통과하면 이 줄이 나온다


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · 실행 에러 ② — 함수가 모르는 이름</font></h3></td></tr></table>

보고서를 파일로 저장하는 함수가 멈춥니다. 글자는 하나도 틀리지 않았습니다. 함수가 날짜를 **받도록** 고치고, 저장한 파일 이름과 파일이 실제로 생겼는지 `assert` 두 줄로 확인하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[완료] daily_report_20261008.md 저장` 그리고 `[테스트 통과] save_report 2건` |

**💡 힌트**

1. 마지막 줄은 `NameError: name 'today' is not defined` 이고, 멈춘 줄은 **함수 안**입니다.
2. 함수 안에서 쓰는 값은 함수 안에서 만들거나 **매개변수로 받아야** 합니다(9/23 오후 함수). 함수 줄과 부르는 줄, 두 곳을 고칩니다.
3. 파일이 있는지는 `os.path.exists(파일이름)` 으로 봅니다(10/2 오후). 결과가 `True` 이면 있습니다.


In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구


def save_report(text):                                        # 보고서 글을 날짜가 든 파일로 저장하는 함수
    filename = f"daily_report_{today}.md"                     # 저장할 파일 이름
    with open(filename, "w", encoding="utf-8") as f:          # 쓰기 모드로 연다
        f.write(text)                                         # 보고서 글을 쓴다
    return filename                                           # 저장한 파일 이름을 리턴한다


saved = save_report("# 야간 보안 관제 보고\n")                  # 보고서를 저장한다
print(f"[완료] {saved} 저장")                                    # 저장한 파일 이름을 알린다

# 1. 이 셀을 그대로 실행하고, 트레이스백에서 멈춘 줄을 찾으세요
# 2. save_report 가 today 를 매개변수로 받게 고치고, 부르는 줄에서 "20261008" 을 넘기세요
# 3. saved 가 "daily_report_20261008.md" 인지, 그 파일이 실제로 있는지 assert 두 줄로 확인하세요


print("[테스트 통과] save_report 2건")                              # assert 가 모두 통과하면 이 줄이 나온다


#### 🐍 문법 상자 · `print` 로 실제 값 찍어 보기 — 대괄호로 감싸면 공백도 보인다

```python
level = " high"
print("[디버그]", level)        # [디버그]  high   — 앞의 공백이 잘 안 보인다
print(f"[디버그] [{level}]")    # [디버그] [ high]  — 대괄호 안을 보면 공백이 보인다
```

| 할 일 | 쓰는 것 |
|---|---|
| 의심 가는 줄 바로 위에 값을 찍는다 | `print(f"[디버그] [{변수}]")` |
| 기대한 값과 글자 하나씩 비교한다 | `[high]` 를 기대했는데 `[High]` · `[ high]` 가 찍혔다 |
| 고친 뒤 `assert` 로 다시 생기지 않게 막는다 | `assert high_count == 4, "high 건수가 다르다"` |

⚠ 대소문자는 `.lower()`, 앞뒤 공백은 `.strip()` 으로 맞춥니다(9/28 · 10/7). 둘을 이어 쓸 수 있습니다 — `값.strip().lower()`.


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · 논리 에러 — 원인이 두 가지인 코드</font></h3></td></tr></table>

high 경보가 4건인데 0건이라고 나옵니다. **디버그 줄을 직접 넣어** 실제 값을 찍고, 원인 두 가지를 찾아 고친 뒤 `assert` 로 막으시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[디버그] [medium]` · `[디버그] [High]` · `[디버그] [HIGH]` · `[디버그] [high ]` · `[디버그] [low]` · `[디버그] [ high]` 여섯 줄, `[요약] high 경보 4건`, `[테스트 통과] high 4건` |

**💡 힌트**

1. 디버그 줄은 `if` 줄 **바로 위**에, 대괄호로 감싸 찍습니다(바로 위 문법 상자).
2. 찍힌 여섯 값 가운데 `high` 로 세야 할 것은 넷입니다. 대소문자만 다른 것과 **공백이 붙은 것**이 있습니다.
3. `if` 줄 하나에서 두 원인을 함께 고칩니다.


In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
judgments = [                                                 # LLM 판단 여섯 건 — 모양이 제각각이다
    {"rule": "night_login", "severity": "medium"},
    {"rule": "brute_force", "severity": "High"},
    {"rule": "port_scan", "severity": "HIGH"},
    {"rule": "password_spraying", "severity": "high "},
    {"rule": "new_ip_login", "severity": "low"},
    {"rule": "config_change", "severity": " high"},
]
# ── 여기까지 ──

high_count = 0                                                # high 건수. 0 에서 시작
for j in judgments:                                           # 판단을 하나씩
    # 1. 이 if 줄 바로 위에 디버그 줄을 넣어 j["severity"] 를 대괄호로 감싸 찍으세요

    # 2. 찍힌 값을 보고 원인 두 가지를 찾아 아래 if 줄을 고치세요
    if j["severity"] == "high":                               # high 이면
        high_count = high_count + 1                           # high 하나를 센다

print(f"[요약] high 경보 {high_count}건")                          # 센 결과

# 3. high_count 가 4 인지 assert 로 확인하세요


print("[테스트 통과] high 4건")                                    # assert 가 통과하면 이 줄이 나온다


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-6 · 논리 에러 세 곳 — 고친 곳마다 막기</font></h3></td></tr></table>

보고서를 만드는 코드에 **논리 에러가 세 곳** 있습니다. 규칙은 「high 가 2건 이상이면 경고 줄」, 「위험한 것부터」, 「한 건에 한 줄」 입니다. 세 곳을 찾아 고치고, **고친 곳마다 그 규칙을 확인하는 `assert` 를 한 줄씩** 쓰시오. 어떤 조건으로 확인할지는 직접 정합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `## 한눈에 보기` 아래 `- **주의: high 경보 2건**`, `## 건별 내역` 아래 `[high] F01` · `[high] F04` · `[medium] F06` · `[low] F02` 가 한 줄씩, 그리고 `[테스트 통과] 규칙 3개` |

**💡 힌트**

1. 먼저 그대로 실행해 기대 결과와 다른 곳을 셋 찾습니다.
2. 확인할 값은 `warning` · `lines` 입니다. 「첫 줄이 high 인가」 는 `lines.splitlines()[0]` 으로 첫 줄을 꺼내 봅니다.


In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
summaries = [
    {"id": "F01", "risk_level": "high", "summary": "admin 계정 탈취 의심"},
    {"id": "F02", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "F04", "risk_level": "high", "summary": "방화벽 규칙 무단 수정 의심"},
    {"id": "F06", "risk_level": "medium", "summary": "jung.hw 새 IP 로그인"},
]
# ── 여기까지 ──

# 논리 에러 세 곳을 찾아 고치세요. 실행은 멈추지 않습니다. 기대 결과와 출력을 한 줄씩 비교합니다
high_count = 0
for s in summaries:
    if s["risk_level"] == "high":
        high_count = high_count + 1

warning = ""
if high_count > 2:
    warning = f"- **주의: high 경보 {high_count}건**\n"

lines = ""
for level in ["low", "medium", "high"]:
    for s in summaries:
        if s["risk_level"] == level:
            lines = lines + f"- [{level}] {s['id']} {s['summary']}"

print("## 한눈에 보기\n" + warning + "\n## 건별 내역\n" + lines)

# 고친 곳마다 그 규칙을 확인하는 assert 를 한 줄씩 쓰세요 (세 줄)



print("[테스트 통과] 규칙 3개")                                    # assert 가 모두 통과하면 이 줄이 나온다


### 📋 6교시 한눈에

| 종류 | 보이는 것 | 오늘 고친 것 |
|---|---|---|
| 문법 에러 (syntax error) | `SyntaxError` — 한 줄도 안 돈다. 하나를 고치면 다음 것이 나온다 | `def` 끝의 `:` · 닫지 않은 괄호 |
| 실행 에러 (runtime error) | `KeyError` · `NameError` — 트레이스백이 두 덩어리로, **아래 덩어리**가 멈춘 줄 | 함수 안의 키 철자 · 함수가 받지 않은 이름 |
| 논리 에러 | 에러 없이 틀린 값 | `.strip().lower()` · `>=` · 정렬 순서 · `\n` |

고친 뒤에는 5교시처럼 `assert` 를 붙여 **같은 에러가 다시 생기면 바로 멈추게** 합니다.


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 완성과 회고</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **회고 (retrospective)** | 개발 팀은 회고에 무엇을 적나 |
| **장애 허용 (fault tolerance)** | 한 부분에 장애가 나도 전체가 할 일을 끝내려면 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 회고 →
- 장애 허용 →

### 왜 필요한가

1. 1과목의 마지막 시간입니다. 만든 것을 **처음부터 끝까지 한 번 더** 돌려 봅니다 — 정상일 때, 알림 서버가 꺼졌을 때.
2. 8일 동안 무엇이 어려웠는지는 **오늘이 아니면 잊습니다.** 캡스톤에서 막힐 때 다시 열어 볼 기록을 남깁니다.
3. 마지막으로 오늘의 산출물을 깃허브에 올립니다.

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 알림 서버를 켜고 시작합니다</mark>

1. 오전 3.1 처럼 **`agent_core` 폴더를 오른쪽 클릭 › Open in Integrated Terminal** 을 누릅니다.
2. 💻 터미널에 `python alert_server.py` 를 입력하고, 그 터미널은 그대로 둡니다.
3. 오전에 켠 서버가 아직 떠 있으면 `Address already in use` 가 나옵니다. 그때는 켜진 그대로 쓰면 됩니다.

| 오늘 확인할 것 | 문제 |
|---|---|
| 명령 한 번으로 보고서 → 알림까지 간다 | 7-2 |
| 알림 서버가 꺼져도 보고서는 남는다 | 7-3 |
| 설정만 바꾸면 코드를 고치지 않아도 동작이 바뀐다 | ⭐7-6 |

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
import notifier

config = notifier.load_config("config_broken.json")
print(config is None)
```

막히면 바로 위 `7.1 알림 서버를 켜고 시작합니다` 설명을 다시 봅니다.

In [ ]:
import notifier

config = notifier.load_config("config_broken.json")
print(config is None)

✅ `[설정 오류] 빠진 키: report_folder` · `True` — `run_pipeline` 은 이때 `[중단]` 을 출력하고 더 가지 않습니다

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-2 · 처음부터 끝까지 한 번 돌리기</font></h3></td></tr></table>

`pipeline.run_pipeline` 으로 오늘 경보를 처리하고, 만들어진 보고서의 소제목을 확인하시오. **LLM 을 두 번 부릅니다.** 알림 서버가 켜져 있어야 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[보고서] daily_report_20261008.md 저장 · 사람 확인 필요 1건` 과 `## 한눈에 보기` · `## 총평` · `## 건별 내역 (위험한 것부터)` — 알림 서버 터미널에 `[알림 받음]` |

**💡 힌트**

1. `pipeline.` 뒤에 함수 이름을 붙여 부릅니다.
2. 소제목 고르기는 10/7 오후 7-5 와 같습니다.
3. 건수는 LLM 판단에 따라 다를 수 있습니다.

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

# 1. pipeline.run_pipeline("config.json", "2026-10-08") 을 실행하세요 — LLM 두 번


# 2. "daily_report_20261008.md" 를 읽어 text 에 담고, "## " 로 시작하는 줄만 출력하세요


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-3 · 알림 서버가 꺼졌을 때</font></h3></td></tr></table>

알림 주소만 꺼진 서버로 바꾼 설정으로 파이프라인을 돌리고, 알림은 실패해도 **보고서는 남는지** 확인하시오. **LLM 을 두 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[알림 실패] ConnectionError` · `[보고서] daily_report_20261009.md 저장 …` · `보고서가 남았나: True` |

**💡 힌트**

1. 설정 파일만 바꿨고 코드는 그대로입니다.
2. `[알림 실패]` 는 오전 3교시의 `notify` 가 찍습니다.
3. 알림 하나 실패했다고 보고서까지 잃지 않는 것 — 장애 허용입니다.

In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:              # 설정 파일을 읽기로 연다
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
config["webhook_url"] = "http://127.0.0.1:5999/alert"        # 아무 서버도 없는 주소 — 알림 서버가 꺼진 상황
with open("config_off.json", "w", encoding="utf-8") as f:     # 주소만 바꾼 설정을 새 파일로 저장한다
    json.dump(config, f, ensure_ascii=False, indent=2)        # 설정을 사람이 읽기 좋게 저장한다
# ── 여기까지 ──

# 1. pipeline.run_pipeline("config_off.json", "2026-10-09") 를 실행하세요 — LLM 두 번


# 2. 보고서 "daily_report_20261009.md" 가 남았는지 os.path.exists 로 확인해 출력하세요


In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구

os.makedirs("../docs", exist_ok=True)                         # docs 폴더가 없으면 만든다. 있으면 그냥 지나간다
print("회고를 저장할 폴더:", os.path.abspath("../docs"))       # 실제 위치 — security-agent-toolkit/docs 여야 한다

#### 🐍 문법 상자 · `%%writefile` 로 다른 폴더에 저장하기

```python
%%writefile ../docs/day08_retrospective.md
# 1과목 회고 (이름)
```

| 경로 | 뜻 |
|---|---|
| `day08_retrospective.md` | 지금 폴더(`agent_core`) 안 |
| `../docs/day08_retrospective.md` | 한 칸 위(`security-agent-toolkit`)의 `docs` 폴더 안 |

⚠ `docs` 폴더가 없으면 저장되지 않습니다. 바로 위 셀이 폴더를 먼저 만듭니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-4 · 1과목 회고 쓰기</font></h3></td></tr></table>

바로 위 셀을 실행한 뒤, 아래 셀의 `- ` 뒤를 **내 말로** 채워 실행하시오. `docs/day08_retrospective.md` 로 저장됩니다. 정답은 예시일 뿐입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `Writing ../docs/day08_retrospective.md` — 왼쪽 목록의 `docs` 폴더에 파일이 생깁니다 |

**💡 힌트**

1. 제목의 `(이름)` 을 내 이름으로 바꿉니다.
2. 「리뷰에서 찾은 것」 은 5교시 5-2 · 6교시에서 찾은 것을 씁니다.
3. 한 항목에 한두 줄이면 충분합니다.

In [ ]:
%%writefile ../docs/day08_retrospective.md
# 1과목 회고 (이름)

## 내가 만든 것
-

## 가장 어려웠던 것
-

## 리뷰에서 찾은 것과 고친 것
-

## 캡스톤에서 더 해 보고 싶은 것
-


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-5 · 오늘의 산출물을 깃허브에 올리기</font></h3></td></tr></table>

**아침 과제 7 에서 `git push` 를 해 본 사람은 명령어로**, 아직이면 **깃허브 웹 화면으로** 올리시오. 코드 셀은 없습니다.

올릴 것: `agent_core` 의 `config.json` · `notifier.py` · `pipeline.py` · `test_agent_core.py` · `daily_report_20261008.md` 와 `docs/day08_retrospective.md`

| | |
|---|---|
| 🎯 나와야 하는 결과 | 브라우저의 내 저장소에서 `agent_core` 와 `docs` 안에 위 파일이 보인다 |

**💡 힌트**

1. 명령어로 올릴 때는 `security-agent-toolkit` 폴더에서 `status` → `add` → `commit` → `push` 순서입니다.
2. `git status` 에 `.env` 가 보이면 올리지 않고 멈춥니다.
3. 웹 화면으로 올릴 때는 그 폴더 안으로 들어가서 **Add file › Upload files** 를 누릅니다.

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-6 · 설정만 바꿔 다시 돌리기</font></h3></td></tr></table>

`approve_severity` 를 `low` 로 바꾼 설정으로 파이프라인을 돌려, 「사람 확인 필요」 건수가 어떻게 바뀌는지 보시오. **코드는 고치지 않습니다. LLM 을 두 번 부릅니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[보고서] daily_report_20261008.md 저장 · 사람 확인 필요 6건` — 기준이 low 라 모든 경보를 사람이 봅니다 |

**💡 힌트**

1. 오전 2-6 과 같은 방법으로 새 설정 파일을 만듭니다.
2. `run_pipeline` 에 새 파일 이름을 넘깁니다.
3. 기준을 낮출수록 사람이 볼 건수가 늘어납니다. 캡스톤에서 고객사가 바뀌면 이 파일만 바꿉니다.

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

# 1. config.json 을 읽어 approve_severity 를 "low" 로 바꾸고 "config_low.json" 으로 저장하세요
# 2. pipeline.run_pipeline("config_low.json", "2026-10-08") 을 실행하세요 — LLM 두 번
# 3. 문제 7-2 의 「사람 확인 필요」 건수와 비교해 보세요


### 📋 7교시 한눈에

| 확인한 것 | 어떻게 |
|---|---|
| 명령 한 번으로 끝까지 | `pipeline.run_pipeline("config.json", 날짜)` |
| 알림이 실패해도 보고서는 남는다 | `config_off.json` — 주소만 바꿨다 |
| 설정이 동작을 바꾼다 | `config_low.json` — 기준만 바꿨다 |
| 기록 | `docs/day08_retrospective.md` |

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 1과목 마무리</mark>

새로 배우는 것은 없습니다. 아래 셀을 실행하면 1과목의 산출물이 있는지 알려 줍니다.

In [ ]:
import os                                                     # 파일이 있는지 보는 도구

# 1과목 산출물 목록
names = ["llm_client.py", "event_summarizer.py", "report_generator.py", "notifier.py", "pipeline.py",
         "test_agent_core.py", "config.json", "daily_report_20261008.md", "../docs/day08_retrospective.md"]

for name in names:                                            # 산출물을 하나씩
    if os.path.exists(name):                                  # 그 파일이 있으면
        print("있음 :", name)                                   # 있다
    else:                                                     # 아니면
        print("없음 :", name, "— 그 파일을 만드는 문제로 돌아갑니다")          # 없다 — 어느 문제로 돌아갈지 알린다

### 1과목에서 만든 것

| 파일 | 하는 일 | 만든 날 |
|---|---|---|
| `llm_client.py` | LLM 을 부르고 답을 JSON 으로 읽는다 | 10/6 |
| `event_summarizer.py` | 경보를 묶음으로 요약하고 위험도순으로 정렬한다 | 10/7 |
| `report_generator.py` | 총평을 받고 보고서를 만든다 | 10/7 |
| `config.json` | 모델 · 정책 · 폴더 · 알림 주소 | 10/8 |
| `notifier.py` | 설정을 검사하며 읽고, 알림을 보낸다. 실패해도 멈추지 않는다 | 10/8 |
| `pipeline.py` | 위 모두를 명령 한 번으로 잇는다 | 10/8 |
| `test_agent_core.py` | 고칠 때마다 망가진 곳이 없는지 본다 | 10/8 |

이 `agent_core` 폴더는 캡스톤에서 그대로 다시 씁니다. 다음 주 월요일(10/12)부터는 2과목 `network_zt` — **네트워크 · 제로 트러스트 운영**을 배웁니다.

### 남은 도전 문제

| 도전 | 무엇을 해 보나 |
|---|---|
| ⭐5-6 | 테스트를 파일 하나로 모아 터미널에서 돌리기 |
| ⭐6-6 | 논리 에러 세 곳 |
| ⭐7-6 | 설정만 바꿔 다시 돌리기 |

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 5-2

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
checklist = ["비밀", "설정", "검증", "실패", "기록"]           # 위 표의 다섯 항목
# ── 여기까지 ──

# 1. review 라는 딕셔너리를 만들어, 다섯 항목마다 "통과" 또는 "위반" 을 담으세요
# (나) 모델 이름 · (다) timeout 없음 · (마) pass 가 위반
review = {"비밀": "통과", "설정": "위반", "검증": "통과", "실패": "위반", "기록": "위반"}

for item in checklist:                                        # 체크리스트 항목을 하나씩
    print(item, review[item])                                 # 항목과 판정

# 2. 「위반」 이 몇 개인지 세어 「위반 3개」 처럼 출력하세요
count = 0                                                     # 센 횟수. 0 에서 시작
for item in checklist:                                        # 체크리스트 항목을 하나씩
    if review[item] == "위반":                                  # 위반이면
        count = count + 1                                     # 하나 센다
print("위반", count, "개")                                       # 위반 수

### 정답 5-3

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다

importlib.reload(report_generator)                            # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
sample = [{"id": "E01", "risk_level": "High", "summary": "로그인 실패 4회"}]   # 대문자가 섞인 요약 한 건
two = [{"id": "E02", "risk_level": "low", "summary": "새 IP 로그인"}, {"id": "E03", "risk_level": "medium", "summary": "심야 접속"}]   # 요약 두 건
# ── 여기까지 ──

# 1. report_generator.make_lines(sample) 이 "- [HIGH] E01 로그인 실패 4회\n" 과 같은지 assert 로 확인하세요. 메시지는 "make_lines 결과가 다르다"
# 대문자 위험도 · id · 요약 · 줄바꿈까지 같아야 한다
assert report_generator.make_lines(sample) == "- [HIGH] E01 로그인 실패 4회\n", "make_lines 결과가 다르다"

# 2. 빈 리스트 [] 를 넣으면 빈 문자열 "" 이 나오는지 assert 로 확인하세요. 메시지는 "빈 리스트는 빈 문자열이어야 한다"
# 요약이 없으면 줄도 없어야 한다
assert report_generator.make_lines([]) == "", "빈 리스트는 빈 문자열이어야 한다"

# 3. 요약 두 건 two 를 넣으면 줄이 두 개(\n 이 두 개)인지 assert 로 확인하세요. 메시지는 "한 건에 한 줄이어야 한다"
assert report_generator.make_lines(two).count("\n") == 2, "한 건에 한 줄이어야 한다"   # 두 건이면 두 줄

print("[테스트 통과] make_lines 3건")                               # 여기까지 오면 assert 가 모두 참이었다

### 정답 5-4

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다

importlib.reload(notifier)                                    # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
config = {"approve_severity": "high"}                       # 기준만 있으면 판정할 수 있다
# ── 여기까지 ──

# 1. needs_approval("high", config) 가 True 인지 assert 로 확인하세요
# high 는 사람에게 묻는다
assert notifier.needs_approval("high", config) == True, "high 는 확인 대상이어야 한다"

# 2. needs_approval("low", config) 가 False 인지 assert 로 확인하세요
# low 는 묻지 않는다
assert notifier.needs_approval("low", config) == False, "low 는 확인 대상이 아니어야 한다"

# 3. 처음 보는 위험도 "critical" 이 True 인지 assert 로 확인하세요
# 처음 보는 위험도도 묻는다 — 모를 때는 막는 쪽
assert notifier.needs_approval("critical", config) == True, "모르는 위험도도 확인 대상이어야 한다"

print("[테스트 통과] needs_approval 3건")                           # 여기까지 오면 assert 가 모두 참이었다

### 정답 5-5

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)                                  # 파일을 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
fenced = "```json\n{\"tool\": \"lock_account\"}\n```"          # LLM 이 자주 보내는 모양 — 코드 블록에 싸인 JSON
# ── 여기까지 ──

# 1. parse_llm_json(fenced) 가 {"tool": "lock_account"} 와 같은지 assert 로 확인하세요
# 코드 블록을 벗기고 딕셔너리로 읽어야 한다
assert llm_client.parse_llm_json(fenced) == {"tool": "lock_account"}, "코드 블록을 벗기지 못했다"

# 2. parse_llm_json("그럴듯한 문장입니다") 가 None 인지 assert 로 확인하세요 (is None)
# JSON 이 아닌 답은 None
assert llm_client.parse_llm_json("그럴듯한 문장입니다") is None, "깨진 입력은 None 이어야 한다"

# 3. parse_llm_json("") 가 None 인지 assert 로 확인하세요
# 빈 답도 None
assert llm_client.parse_llm_json("") is None, "빈 답은 None 이어야 한다"

print("[테스트 통과] parse_llm_json 3건")                           # 여기까지 오면 assert 가 모두 참이었다

### 정답 ⭐5-6

In [ ]:
%%writefile test_agent_core.py
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다
import notifier                                               # 3교시에 만든 notifier.py 를 불러온다
import report_generator                                       # 오후에 만든 report_generator.py 를 불러온다

# 1. 문제 5-3 의 assert 세 줄을 옮기세요 (sample · two 도 함께)
# 대문자가 섞인 요약 한 건
sample = [{"id": "E01", "risk_level": "High", "summary": "로그인 실패 4회"}]
# 대문자 위험도 · id · 요약 · 줄바꿈까지 같아야 한다
assert report_generator.make_lines(sample) == "- [HIGH] E01 로그인 실패 4회\n", "make_lines 결과가 다르다"
# 요약이 없으면 줄도 없어야 한다
assert report_generator.make_lines([]) == "", "빈 리스트는 빈 문자열이어야 한다"
two = [{"id": "E02", "risk_level": "low", "summary": "새 IP 로그인"}, {"id": "E03", "risk_level": "medium", "summary": "심야 접속"}]   # 요약 두 건
assert report_generator.make_lines(two).count("\n") == 2, "한 건에 한 줄이어야 한다"   # 두 건이면 두 줄

# 2. 문제 5-4 의 assert 세 줄을 옮기세요 (config 도 함께)
config = {"approve_severity": "high"}                         # 기준만 있으면 판정할 수 있다
# high 는 사람에게 묻는다
assert notifier.needs_approval("high", config) == True, "high 는 확인 대상이어야 한다"
# low 는 묻지 않는다
assert notifier.needs_approval("low", config) == False, "low 는 확인 대상이 아니어야 한다"
# 처음 보는 위험도도 묻는다 — 모를 때는 막는 쪽
assert notifier.needs_approval("critical", config) == True, "모르는 위험도도 확인 대상이어야 한다"

# 3. 문제 5-5 의 assert 세 줄을 옮기세요 (fenced 도 함께)
fenced = "```json\n{\"tool\": \"lock_account\"}\n```"         # 코드 블록에 싸인 JSON
# 코드 블록을 벗기고 딕셔너리로 읽어야 한다
assert llm_client.parse_llm_json(fenced) == {"tool": "lock_account"}, "코드 블록을 벗기지 못했다"
# JSON 이 아닌 답은 None
assert llm_client.parse_llm_json("그럴듯한 문장입니다") is None, "깨진 입력은 None 이어야 한다"
# 빈 답도 None
assert llm_client.parse_llm_json("") is None, "빈 답은 None 이어야 한다"

print("[테스트 통과] 9건 모두")                                       # 여기까지 오면 아홉 줄이 모두 참이었다

💻 **터미널에 입력합니다** — `agent_core` 폴더에서 연 터미널입니다.

```bash
python test_agent_core.py
```

### 정답 6-2

In [ ]:
# 고친 곳 1 — SyntaxError: expected ':' — def 줄 끝에 : 이 빠졌다
def make_title(today):                                        # 날짜가 든 보고서 제목을 만드는 함수
    return f"# 야간 보안 관제 보고 ({today})"


def make_filename(today):                                     # 날짜로 보고서 파일 이름을 만드는 함수
    return "daily_report_" + today.replace("-", "") + ".md"   # 2026-10-08 → daily_report_20261008.md


print(make_title("2026-10-08"))                               # 고친 곳 2 — '(' was never closed — 닫는 괄호

assert make_title("2026-10-08") == "# 야간 보안 관제 보고 (2026-10-08)", "제목이 다르다"      # 제목 확인
assert make_filename("2026-10-08") == "daily_report_20261008.md", "파일 이름이 다르다"   # 파일 이름 확인

print("[테스트 통과] 제목 · 파일 이름 2건")                        # assert 가 모두 통과하면 이 줄이 나온다


### 정답 6-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
judgments = [                                                 # LLM 판단 두 건이라고 가정합니다
    {"rule": "brute_force", "severity": "high", "tool": "lock_account"},
    {"rule": "night_login", "severity": "medium", "tool": "notify_only"},
]
# ── 여기까지 ──


def make_line(j):                                             # 판단 한 건을 보고서 한 줄로 바꾸는 함수
    return f"- [{j['severity']}] {j['rule']} → {j['tool']}"    # 고친 곳 — 트레이스백 아래 덩어리의 줄. 'sevrity' → 'severity'


lines = ""                                                    # 보고서 줄을 모을 빈 문자열
for j in judgments:                                           # 판단을 하나씩
    lines = lines + make_line(j) + "\n"                       # 한 줄씩 이어 붙인다
print(lines)                                                  # 모은 줄을 출력한다

assert make_line(judgments[0]) == "- [high] brute_force → lock_account", "첫 줄이 다르다"   # 한 건이 맞게 바뀌나
assert lines.count("\n") == 2, "줄 수가 다르다"                 # 두 건이 두 줄이 됐나

print("[테스트 통과] make_line 2건")                                # assert 가 모두 통과하면 이 줄이 나온다


### 정답 6-4

In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구


def save_report(text, today):                                 # 고친 곳 1 — 날짜를 매개변수로 받는다
    filename = f"daily_report_{today}.md"                     # 저장할 파일 이름
    with open(filename, "w", encoding="utf-8") as f:          # 쓰기 모드로 연다
        f.write(text)                                         # 보고서 글을 쓴다
    return filename                                           # 저장한 파일 이름을 리턴한다


saved = save_report("# 야간 보안 관제 보고\n", "20261008")        # 고친 곳 2 — 부를 때 날짜를 넘긴다
print(f"[완료] {saved} 저장")                                    # 저장한 파일 이름을 알린다

assert saved == "daily_report_20261008.md", "파일 이름이 다르다"    # 이름이 맞나
assert os.path.exists(saved), "파일이 만들어지지 않았다"               # 파일이 실제로 생겼나

print("[테스트 통과] save_report 2건")                              # assert 가 모두 통과하면 이 줄이 나온다


### 정답 6-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
judgments = [                                                 # LLM 판단 여섯 건 — 모양이 제각각이다
    {"rule": "night_login", "severity": "medium"},
    {"rule": "brute_force", "severity": "High"},
    {"rule": "port_scan", "severity": "HIGH"},
    {"rule": "password_spraying", "severity": "high "},
    {"rule": "new_ip_login", "severity": "low"},
    {"rule": "config_change", "severity": " high"},
]
# ── 여기까지 ──

high_count = 0                                                # high 건수. 0 에서 시작
for j in judgments:                                           # 판단을 하나씩
    print(f"[디버그] [{j['severity']}]")                         # 대괄호로 감싸 찍는다 — 공백까지 보인다
    if j["severity"].strip().lower() == "high":               # 원인 두 가지 — 앞뒤 공백은 strip, 대소문자는 lower
        high_count = high_count + 1                           # high 하나를 센다

print(f"[요약] high 경보 {high_count}건")                          # 센 결과

assert high_count == 4, "high 건수가 다르다"                     # 같은 실수가 다시 생기면 여기서 멈춘다

print("[테스트 통과] high 4건")                                    # assert 가 통과하면 이 줄이 나온다


### 정답 ⭐6-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
summaries = [                                                 # 미리 준비한 요약 — 모두 같은 결과를 보도록 고정했다
    {"id": "F01", "risk_level": "high", "summary": "admin 계정 탈취 의심"},
    {"id": "F02", "risk_level": "low", "summary": "kim01 로그인 실패 1회"},
    {"id": "F04", "risk_level": "high", "summary": "방화벽 규칙 무단 수정 의심"},
    {"id": "F06", "risk_level": "medium", "summary": "jung.hw 새 IP 로그인"},
]
# ── 여기까지 ──

high_count = 0                                                # high 건수. 0 에서 시작
for s in summaries:                                           # 요약을 하나씩 꺼낸다
    if s["risk_level"] == "high":                             # high 이면
        high_count = high_count + 1                           # high 하나를 센다

warning = ""                                                  # 기본은 경고 없음
if high_count >= 2:                                           # 고친 곳 1 — 「2건 이상」 이므로 >=
    warning = f"- **주의: high 경보 {high_count}건**\n"            # 경고 줄

lines = ""                                                    # 목록 줄을 모을 빈 문자열
for level in ["high", "medium", "low"]:                       # 고친 곳 2 — 위험한 것부터
    for s in summaries:                                       # 요약을 하나씩 꺼낸다
        if s["risk_level"] == level:                          # 지금 위험도와 같으면
            lines = lines + f"- [{level}] {s['id']} {s['summary']}\n"   # 고친 곳 3 — 줄 끝의 \n

print("## 한눈에 보기\n" + warning + "\n## 건별 내역\n" + lines)       # 작은 보고서를 출력한다

assert warning != "", "high 가 2건인데 경고 줄이 없다"                # 규칙 1 — 2건 이상이면 경고
assert lines.splitlines()[0].startswith("- [high]"), "첫 줄이 high 가 아니다"   # 규칙 2 — 위험한 것부터
assert lines.count("\n") == len(summaries), "한 건에 한 줄이 아니다"   # 규칙 3 — 한 건에 한 줄

print("[테스트 통과] 규칙 3개")                                    # assert 가 모두 통과하면 이 줄이 나온다


### 정답 7-2

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

# 1. pipeline.run_pipeline("config.json", "2026-10-08") 을 실행하세요 — LLM 두 번
pipeline.run_pipeline("config.json", "2026-10-08")            # 처음부터 끝까지 — LLM 두 번

# 2. "daily_report_20261008.md" 를 읽어 text 에 담고, "## " 로 시작하는 줄만 출력하세요
with open("daily_report_20261008.md", encoding="utf-8") as f:  # 만들어진 보고서를 읽기로 연다
    text = f.read()                                           # 파일 전체를 문자열 하나로 읽는다
for line in text.split("\n"):                                 # 한 줄씩 나눠 본다
    if line.startswith("## "):                                # 소제목 줄이면
        print(line)                                           # 그 줄을 출력한다

### 정답 7-3

In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
with open("config.json", encoding="utf-8") as f:              # 설정 파일을 읽기로 연다
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
config["webhook_url"] = "http://127.0.0.1:5999/alert"        # 아무 서버도 없는 주소 — 알림 서버가 꺼진 상황
with open("config_off.json", "w", encoding="utf-8") as f:     # 주소만 바꾼 설정을 새 파일로 저장한다
    json.dump(config, f, ensure_ascii=False, indent=2)        # 설정을 사람이 읽기 좋게 저장한다
# ── 여기까지 ──

# 1. pipeline.run_pipeline("config_off.json", "2026-10-09") 를 실행하세요 — LLM 두 번
pipeline.run_pipeline("config_off.json", "2026-10-09")        # 꺼진 서버 설정으로 — LLM 두 번

# 2. 보고서 "daily_report_20261009.md" 가 남았는지 os.path.exists 로 확인해 출력하세요
print("보고서가 남았나:", os.path.exists("daily_report_20261009.md"))  # 알림은 실패해도 파일은 있어야 한다

### 정답 7-4

In [ ]:
%%writefile ../docs/day08_retrospective.md
# 1과목 회고 (홍길동)

## 내가 만든 것
- 경보를 묶음으로 요약하고, 위험도순 보고서를 만들어 저장한 뒤 알림까지 보내는 파이프라인 — 명령 한 번(python pipeline.py)으로 돈다

## 가장 어려웠던 것
- 함수 사이로 값 넘기기. print 만 하고 return 을 안 해서 다음 함수가 None 을 받았다

## 리뷰에서 찾은 것과 고친 것
- 모델 이름이 코드에 박혀 있었다 → config.json 으로 뺐다
- 요청에 timeout 이 없었다 → timeout=5 와 try/except 를 넣었다

## 캡스톤에서 더 해 보고 싶은 것
- 알림을 진짜 메신저로 보내기, 도구를 하나 더 붙이기

### 정답 7-5

**`git push` 를 아는 사람** — 💻 터미널에 입력합니다. `agent_core` 에서 연 터미널이면 먼저 한 칸 위로 올라갑니다.

```bash
cd ..
git status
git add agent_core docs
git commit -m "Add day 8 pipeline, tests and retrospective"
git push
```

`git status` 에 `.env` 가 보이면 `add` 하지 않고 멈춥니다(아침 과제 7 의 6번).

**아직 모르는 사람** — 브라우저에서 내 `security-agent-toolkit` 저장소를 엽니다.

1. `agent_core` 폴더로 들어가 **Add file › Upload files** 로 `config.json` · `notifier.py` · `pipeline.py` · `test_agent_core.py` · `daily_report_20261008.md` 를 끌어다 놓고 **Commit changes** 를 누릅니다.
2. 저장소 첫 화면으로 돌아와 `docs` 폴더로 들어가 같은 방법으로 `day08_retrospective.md` 를 올립니다.

### 정답 ⭐7-6

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import pipeline                                               # pipeline.py 를 불러온다 — 실행은 되지 않아야 한다

importlib.reload(pipeline)                                    # pipeline.py 를 고쳤으면 다시 읽는다

# 1. config.json 을 읽어 approve_severity 를 "low" 로 바꾸고 "config_low.json" 으로 저장하세요
# 2. pipeline.run_pipeline("config_low.json", "2026-10-08") 을 실행하세요 — LLM 두 번
# 3. 문제 7-2 의 「사람 확인 필요」 건수와 비교해 보세요
with open("config.json", encoding="utf-8") as f:              # 설정 파일을 읽기로 연다
    config = json.load(f)                                     # 설정을 딕셔너리로 읽는다
config["approve_severity"] = "low"                            # 기준을 low 로 낮춘다
with open("config_low.json", "w", encoding="utf-8") as f:     # 새 설정 파일로 저장한다 — config.json 은 그대로
    json.dump(config, f, ensure_ascii=False, indent=2)        # 설정을 사람이 읽기 좋게 저장한다

pipeline.run_pipeline("config_low.json", "2026-10-08")        # 바뀐 설정으로 — LLM 두 번